In [ ]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

# BPE FOR GPT-2 MODEL

In [ ]:
print(tokenizer.n_vocab) 

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

In [ ]:
strings = tokenizer.decode(integers)

print(strings)

**Let us take another simple example to illustrate how the BPE tokenizer deals with unknown tokens**

In [ ]:
integers = tokenizer.encode("Akwirw ier")
print(integers)

strings = tokenizer.decode(integers)
print(strings)

 ### CREATING INPUT-TARGET PAIRS

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))  # vocab size for this verdict dataset after applying BPE
# Executing the code above will return 5145, the total number of tokens in the training set, after applying the BPE tokenizer.

In [ ]:
enc_sample = enc_text[50:]
# Next, we remove the first 50 tokens from the dataset for demonstration purposes as it
# results in a slightly more interesting text passage in the next steps

**One of the easiest and most intuitive ways to create the input-target pairs for the nextword prediction task is to create two variables, x and y, where x contains the input tokens
and y contains the targets, which are the inputs shifted by 1**

In [ ]:
## The context size determines how many tokens are included in the input

In [ ]:
context_size = 4 #length of the input
#The context_size of 4 means that the model is trained to look at a sequence of 4 words (or tokens) 
#to predict the next word in the sequence. 
#The input x is the first 4 tokens [1, 2, 3, 4], and the target y is the next 4 tokens [2, 3, 4, 5]

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

In [ ]:
# For illustration purposes, let's repeat the previous code but convert the token IDs into text
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

**Implementing an efficient data loader that
iterates over the input dataset and returns the inputs and targets as PyTorch tensors, which
can be thought of as multidimensional arrays.**

### IMPLEMENTING A DATA LOADER

In [ ]:
# Understanding DataSet and DataLoader in PyTorch

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# 1. THE DATASET (The Vending Machine)
class FruitDataset(Dataset):
    def __init__(self):
        self.fruits = ["Apple", "Banana", "Cherry", "Date", "Elderberry"]
        self.colors = ["Red", "Yellow", "Red", "Brown", "Purple"]

    def __len__(self):
        return len(self.fruits)

    def __getitem__(self, idx):
        # Fetch one item for the index given by the Sampler
        return self.fruits[idx], self.colors[idx]

# 2. THE DATALOADER (The Delivery Person)
my_dataset = FruitDataset()
# Setting batch_size=2 and shuffle=True
loader = DataLoader(my_dataset, batch_size=2, shuffle=True)

# 3. THE USE CASE (The Training Loop)
print("--- Starting Delivery ---")
for i, (fruits, colors) in enumerate(loader):
    print(f"Batch {i+1} delivered: {fruits} (Colors: {colors})")

**For the efficient data loader implementation, we will use PyTorch's built-in Dataset and DataLoader classes**

In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

**The following code will use the GPTDatasetV1 to load the inputs in batches via a PyTorch**

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

**Testing the dataloader with a batch size of 1 for an LLM with a context size of 4**

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

**To illustrate the meaning of stride=1**

In [ ]:
second_batch = next(data_iter)
print(second_batch)

**Batch sizes of 1, such as we have sampled from the data loader so far, are useful for illustration purposes.**

**If you have previous experience with deep learning, you may know that small batch sizes require less memory during training but lead to more noisy model updates.**

**Just like in regular deep learning, the batch size is a trade-off and hyperparameter to experiment with when training LLMs.**

In [ ]:
# data loader to sample with a batch size greater than 1:

In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

In [ ]:
num_batches = len(dataloader)
print(f"Total batches: {num_batches}")

In [ ]:
#printing all input output paits of batch size 8 and  context size 4 totally they are 160 batches as such
for i, (input_pair, target_pair) in enumerate(dataloader):
    print(f"\nBatch {i+1}")
    for inp, tgt in zip(input_pair, target_pair):
        print(f"{str(inp):<30} -> {tgt}")

In [ ]:
for i, (input_pair, target_pair) in enumerate(dataloader):
    print(f"Batch {i+1} {input_pair},{target_pair}")

**Note that we increase the stride to 4. This is to utilize the data set fully (we don't skip a
single word) but also avoid any overlap between the batches, since more overlap could lead
to increased overfitting.**

# Vector Embeddings

In [ ]:
# firstly understanding about basics of vector embeddings
# Import trained model
# !pip install gensim


In [ ]:
# import gensim.downloader as api
# model = api.load("word2vec-google-news-300")  # download the model and return as object ready for use

In [ ]:
# # Example of a word as a vector
# word_vectors=model

# # Let us look how the vector embedding of a word looks like
# print(word_vectors['computer'])  # Example: Accessing the vector for the word 'computer'

In [ ]:
# # Similar words
# # Example of using most_similar
# print(word_vectors.most_similar(positive=['king', 'woman'], negative=['man'], topn=10))


In [ ]:
# # Let us check the similarity b/w a few pair of words
# # Example of calculating similarity
# print(word_vectors.similarity('woman', 'man'))
# print(word_vectors.similarity('king', 'queen'))
# print(word_vectors.similarity('uncle', 'aunt'))
# print(word_vectors.similarity('boy', 'girl'))
# print(word_vectors.similarity('nephew', 'niece'))
# print(word_vectors.similarity('paper', 'water'))

In [ ]:
# import numpy as np
# # Words to compare
# word1 = 'man'
# word2 = 'woman'

# word3 = 'semiconductor'
# word4 = 'earthworm'

# word5 = 'nephew'
# word6 = 'niece'

# # Calculate the vector difference
# vector_difference1 = model[word1] - model[word2]
# vector_difference2 = model[word3] - model[word4]
# vector_difference3 = model[word5] - model[word6]

# # Calculate the magnitude of the vector difference
# magnitude_of_difference1 = np.linalg.norm(vector_difference1)
# magnitude_of_difference2 = np.linalg.norm(vector_difference2)
# magnitude_of_difference3 = np.linalg.norm(vector_difference3)


# # Print the magnitude of the difference
# print("The magnitude of the difference between '{}' and '{}' is {:.2f}".format(word1, word2, magnitude_of_difference1))
# print("The magnitude of the difference between '{}' and '{}' is {:.2f}".format(word3, word4, magnitude_of_difference2))
# print("The magnitude of the difference between '{}' and '{}' is {:.2f}".format(word5, word6, magnitude_of_difference3))

### CREATING TOKEN EMBEDDINGS

In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [ ]:
print(embedding_layer.weight)

**We can see that the weight matrix of the embedding layer contains small, random values. These values are optimized during LLM training as part of the LLM optimization itself,Moreover, we can see that the weight matrix has six rows and three columns. There is one row for each of the six possible tokens in the vocabulary. And there is one column for each of the three embedding dimensions.**

In [ ]:
print(embedding_layer(torch.tensor([3])))

In [ ]:
print(embedding_layer(input_ids))


### POSITIONAL EMBEDDINGS (ENCODING WORD POSITIONS)

In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [ ]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [ ]:
print(len(raw_text))

In [ ]:
print(len(dataloader)) # number batches each batch size is 4
print(160*8*4) # total number of tokens in raw_text

In [ ]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

In [ ]:
print("Token IDs:\n", targets)
print("\ntargets shape:\n", targets.shape)

In [ ]:
print(type(token_embedding_layer))


In [ ]:
print(token_embedding_layer.weight)

In [ ]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

In [ ]:
print(token_embeddings)

**For a GPT model's absolute embedding approach, we just need to create another
embedding layer that has the same dimension as the token_embedding_layer**

In [ ]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [ ]:
print(pos_embedding_layer)

In [ ]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

In [ ]:
print(pos_embeddings)

In [ ]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

In [ ]:
print(input_embeddings)

**input_embeddings we created are the embedded input examples that can now be processed by the main LLM modules**

## IMPLEMENTING A SIMPLIFIED ATTENTION MECHANISM

In [ ]:
# Consider the following input sentence, which has already been embedded into 3- dimensional vectors.
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [ ]:
# import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d import Axes3D

# # Corresponding words
# words = ['Your', 'journey', 'starts', 'with', 'one', 'step']

# # Extract x, y, z coordinates
# x_coords = inputs[:, 0].numpy()
# y_coords = inputs[:, 1].numpy()
# z_coords = inputs[:, 2].numpy()

# # Create 3D plot
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')

# # Plot each point and annotate with corresponding word
# for x, y, z, word in zip(x_coords, y_coords, z_coords, words):
#     ax.scatter(x, y, z)
#     ax.text(x, y, z, word, fontsize=10)

# # Set labels for axes
# ax.set_xlabel('X')
# ax.set_ylabel('Y')
# ax.set_zlabel('Z')

# plt.title('3D Plot of Word Embeddings')
# plt.show()

In [ ]:
print(inputs)

In [ ]:
#Each row represents a word, and each column represents an embedding dimension
# The second input token serves as the query
query = inputs[1]  # 2nd input token is the query

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) # dot product (transpose not necessary here since they are 1-dim vectors)

print(attn_scores_2)

**In the next step, we normalize each of the attention scores that
we computed previously.**

In [ ]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

**In practice, it's more common and advisable to use the softmax function for normalization.**

**This approach is better at managing extreme values and offers more favorable gradient properties during training.** 

**Below is a basic implementation of the softmax function for normalizing the attention scores**

In [ ]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

**Note that this naive softmax implementation (softmax_naive) may encounter numerical instability problems, such as overflow and underflow, when dealing with large or small input values.**

**Therefore, in practice, it's advisable to use the PyTorch implementation of softmax, which has been extensively optimized for performance**

In [ ]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

**The context vector z(2)is calculated as a weighted sum of all input
vectors.** 

**This involves multiplying each input vector by its corresponding attention weight.**


In [ ]:
query = inputs[1] # 2nd input token is the query

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)

In [ ]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

**When computing the preceding attention score tensor, we used for-loops in Python.**
                                                            
**However, for-loops are generally slow, and we can achieve the same results using matrix multiplication:**

In [ ]:
attn_scores = inputs @ inputs.T
print(attn_scores)

In [ ]:
# We now normalize each row so that the values in each row sum to 1
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

**In the context of using PyTorch, the dim parameter in functions like torch.softmax specifies the dimension of the input tensor along which the function will be computed.**

**By setting dim=-1, we are instructing the softmax function to apply the normalization along the last dimension of the attn_scores tensor.**

**If attn_scores is a 2D tensor (for example, with a shape of [rows, columns]), dim=-1 will normalize across the columns so that the values in each row (summing over the column dimension) sum up to 1.**

In [ ]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)
print("All row sums:", attn_weights.sum(dim=-1))

**In the third and last step, we now use these attention weights to compute all context vectors via matrix multiplication**

In [ ]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

In [ ]:
print("Previous 2nd context vector:", context_vec_2)

## IMPLEMENTING SELF ATTENTION WITH TRAINABLE WEIGHTS

In [ ]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [ ]:
x_2 = inputs[1] #A
d_in = inputs.shape[1] #B
d_out = 2 #C

**Note that in GPT-like models, the input and output dimensions are usually the same.**

**To better follow the computation, we choose different input (d_in=3) and output (d_out=2) dimensions here.**

**Next, we initialize the three weight matrices Wq, Wk and Wv**

In [ ]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [ ]:
print(W_query)

In [ ]:
print(W_key)

In [ ]:
print(W_value)

**Note that we are setting requires_grad=False to reduce clutter in the outputs for illustration purposes.**

**If we were to use the weight matrices for model training, we would set requires_grad=True to update these matrices during model training.**

In [ ]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

In [ ]:
keys = inputs @ W_key
values = inputs @ W_value
queries = inputs @ W_query
print("keys.shape:", keys.shape)

print("values.shape:", values.shape)

print("queries.shape:", queries.shape)

In [ ]:
keys_2 = keys[1] #A
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

In [ ]:
attn_scores_2 = query_2 @ keys.T # All attention scores for given query
print(attn_scores_2)

In [ ]:
attn_scores = queries @ keys.T # omega
print(attn_scores)

**We compute the attention weights by scaling the attention scores and using the softmax function we used earlier.** 

**The difference to earlier is that we now scale the attention scores by dividing them by the square root of the
embedding dimension of the keys.**

**Note that taking the square root is mathematically the same as exponentiating by 0.5**

In [ ]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)
print(d_k)

## WHY DIVIDE BY SQRT (DIMENSION)

**Reason 1: For stability in learning**

The softmax function is sensitive to the magnitudes of its inputs. When the inputs are large, the differences between the exponential values of each input become much more pronounced. This causes the softmax output to become "peaky," where the highest value receives almost all the probability mass, and the rest receive very little.

In attention mechanisms, particularly in transformers, if the dot products between query and key vectors become too large (like multiplying by 8 in this example), the attention scores can become very large. This results in a very sharp softmax distribution, making the model overly confident in one particular "key." Such sharp distributions can make learning unstable,

In [ ]:
import torch

# Define the tensor
tensor = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])

# Apply softmax without scaling
softmax_result = torch.softmax(tensor, dim=-1)
print("Softmax without scaling:", softmax_result)

# Multiply the tensor by 8 and then apply softmax
scaled_tensor = tensor * 8
softmax_scaled_result = torch.softmax(scaled_tensor, dim=-1)
print("Softmax after scaling (tensor * 8):", softmax_scaled_result)

## BUT WHY SQRT?

**Reason 2: To make the variance of the dot product stable**

The dot product of  Q and K increases the variance because multiplying two random numbers increases the variance.

The increase in variance grows with the dimension. 

Dividing by sqrt (dimension) keeps the variance close to 1

In [ ]:
import numpy as np

# Function to compute variance before and after scaling
def compute_variance(dim, num_trials=1000):
    dot_products = []
    scaled_dot_products = []

    # Generate multiple random vectors and compute dot products
    for _ in range(num_trials):
        q = np.random.randn(dim)
        k = np.random.randn(dim)
        
        # Compute dot product
        dot_product = np.dot(q, k)
        dot_products.append(dot_product)
        
        # Scale the dot product by sqrt(dim)
        scaled_dot_product = dot_product / np.sqrt(dim)
        scaled_dot_products.append(scaled_dot_product)
    
    # Calculate variance of the dot products
    variance_before_scaling = np.var(dot_products)
    variance_after_scaling = np.var(scaled_dot_products)

    return variance_before_scaling, variance_after_scaling

# For dimension 5
variance_before_5, variance_after_5 = compute_variance(5)
print(f"Variance before scaling (dim=5): {variance_before_5}")
print(f"Variance after scaling (dim=5): {variance_after_5}")

# For dimension 20
variance_before_100, variance_after_100 = compute_variance(100)
print(f"Variance before scaling (dim=100): {variance_before_100}")
print(f"Variance after scaling (dim=100): {variance_after_100}")



**We now compute the context vector as a weighted sum over the value vectors.**

**Here, the attention weights serve as a weighting factor that weighs the respective importance of each value vector.We can use matrix multiplication to obtain the output in one step**

In [ ]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

## IMPLEMENTING A COMPACT SELF ATTENTION PYTHON CLASS

In [ ]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        
        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

In [ ]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

**We can improve the SelfAttention_v1 implementation further by utilizing PyTorch nn.Linear layers, which effectively perform matrix multiplication when the bias units are disabled.**

**Additionally, a significant advantage of using nn.Linear instead of manually implementing nn.Parameter(torch.rand(...)) is that nn.Linear has an optimized weight initialization scheme, contributing to more stable and effective model training.**

In [ ]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

In [ ]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

**Note that SelfAttention_v1 and SelfAttention_v2 give different outputs because they use different initial weights for the weight matrices since nn.Linear uses a more sophisticated weight initialization scheme.**

## HIDING FUTURE WORDS WITH CAUSAL ATTENTION

In [ ]:
#Let's work with the attention scores and weights from the previous section to code the causal attention mechanism.

In [ ]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [ ]:
queries = sa_v2.W_query(inputs) #  (shape:(6,2)) d_in = inputs.shape[1](6) d_out = 2
keys = sa_v2.W_key(inputs)   # (shape:(6,2))
attn_scores = queries @ keys.T  # (6,6) as queries(6,2)  keys (2,6)
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=1)
print(attn_weights)

In [ ]:
context_length=6

**We can now use PyTorch's tril function to create a mask where the values above the diagonal are zero**

In [ ]:
torch.ones(context_length, context_length)

In [ ]:
context_length = attn_scores.shape[0] # In  (6,6) in which attn_scores.shape[0] ----> (6) 
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

In [ ]:
#Now, we can multiply this mask with the attention weights to zero out the values above the diagonal(element wise multiplication)
masked_simple = attn_weights*mask_simple
print(masked_simple)

**The third step is to renormalize the attention weights to sum up to 1 again in each row.**

**We can achieve this by dividing each element in each row by the sum in each row**

In [ ]:
row_sums = masked_simple.sum(dim=1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

**While we could be technically done with implementing causal attention at this point, we can take advantage of a mathematical property of the softmax function.** 

**We can implement the computation of the masked attention weights more efficiently in fewer steps.**

In [ ]:
torch.triu(torch.ones(context_length, context_length))

In [ ]:
torch.triu(torch.ones(context_length, context_length),diagonal=1) # when diagonal=1 Main diagonal removed Only values strictly above it

In [ ]:
torch.triu(torch.ones(context_length, context_length),diagonal=2)

In [ ]:
torch.triu(torch.ones(context_length, context_length),diagonal=-1)

In [ ]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)# when diagonal=1 Main diagonal removed Only values strictly above it
print(mask)

In [ ]:
attn_scores

In [ ]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

In [ ]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
print(attn_weights) 
# this is the most efficient way to calculate attention weights than previous way in the above as if we
# first take soft max then apply mask then again re normalize it does not efficent way

**As we can see based on the output, the values in each row sum to 1, and no further normalization is necessary.**

**We could now use the modified attention weights to compute the context vectors via
context_vec = attn_weights @ values.**

**However, in the next section,we first cover another minor tweak to the causal attention mechanism that is useful for
reducing overfitting when training LLMs.**

### MASKING ADDITIONAL ATTENTION WEIGHTS WITH DROPOUT

In [ ]:
example = torch.ones(6, 6) #B
print(example)

In [ ]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) #A
example = torch.ones(6, 6) #B
print(dropout(example))

In [ ]:
torch.manual_seed(123)
print(dropout(attn_weights))

**As we can see above, the resulting attention weight matrix now has additional elements zeroed out and the remaining ones rescaled.**

### IMPLEMENTING A COMPACT CAUSAL ATTENTION CLASS

In [ ]:
inputs

In [ ]:
# 2 inputs with 6 tokens each, and each token has embedding dimension 3
batch = torch.stack((inputs, inputs), dim=0)  # stack vertically
print(batch.shape) 

In [ ]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # New
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, num_tokens, d_in = x.shape # New batch dimension b
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec

In [ ]:
print(d_in)

In [ ]:
print(d_out)

In [ ]:
# batch = torch.stack((inputs, inputs), dim=0)  # stack vertically
# print(batch.shape) 

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1] #(2,6,3) where 2 represent nunmber of batches
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch) 
# callable object ca(bactch represent that it call _call__ of nn module which internally calls forward method of CausalAttention)
print("context_vecs.shape:", context_vecs.shape)

In [ ]:
print(context_vecs) # the resulting context vector is a 3D tensor where each token is now represented by a 2D embedding

**In the next section, we will expand on this concept and implement a multi-head attention module, that implements several of such causal attention mechanisms in parallel.**

## EXTENDING SINGLE HEAD ATTENTION TO MULTI-HEAD ATTENTION

**Implementing multi-head attention involves creating multiple instances of the  causal self-attention mechanism, each with its own weights, and then combining their outputs**

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


**If we use this MultiHeadAttentionWrapper class with two attention heads (via num_heads=2) and CausalAttention output dimension d_out=2, this results in a 4-dimensional context vectors (d_out*num_heads=4)**

In [ ]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) 

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1] # This is the number of tokens = 6
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch) 
# mha(batch) calls mha.__call__(batch)
# __call__ is defined in nn.Module (not __Callable__)
# inside __call__, it invokes self.forward(batch)
# so this ends up calling MultiHeadAttentionWrapper.forward
# since your class inherits from nn.Module, it inherits this behavior  
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

**We implemented a MultiHeadAttentionWrapper that combined multiple single-head attention modules.**

**However, note that these are processed sequentially via[head(x) for head in self.heads] in the forward method.
We can improve this implementation by processing the heads in parallel.**

**One way to achieve this is by computing the outputs for all attention heads simultaneously via matrix multiplication.**

### IMPLEMENTING MULTI-HEAD ATTENTION WITH WEIGHT SPLITS

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) # use linear layer because it is optimized for initializing the weights later useful in back propagation
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1) # dividing by sqrt of head_dim 3 in this case  # dim = -1 such that all columns in single row sum up to 1
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

In [ ]:
torch.manual_seed(123)

# Define the tensor with 3 rows and 6 columns
inputs = torch.tensor(
    [[0.43, 0.15, 0.89, 0.55, 0.87, 0.66],  # Row 1
     [0.57, 0.85, 0.64, 0.22, 0.58, 0.33],  # Row 2
     [0.77, 0.25, 0.10, 0.05, 0.80, 0.55]]  # Row 3
)

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) 

batch_size, context_length, d_in = batch.shape
d_out = 6
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)


**step 1: Reduce the projection dim to match desired output dim**

**Step 2: Use a Linear layer to combine head outputs**

**Step 3: Tensor shape: (b, num_tokens, d_out)**

**Step 4: We implicitly split the matrix by adding a `num_heads` dimension. Then we unroll last dim: (b,
num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)**

**Step 5: Transpose from shape (b, num_tokens, num_heads, head_dim) to (b, num_heads, num_tokens, head_dim)**

**Step 6: Compute dot product for each head**

**Step 7: Mask truncated to the number of tokens**

**Step 8: Use the mask to fill attention scores**

**Step 9: Tensor shape: (b, num_tokens, n_heads, head_dim)**

**Step 10: Combine heads, where self.d_out = self.num_heads * self.head_dim**

**Step 11: Add an optional linear projection**

## DETAILED EXPLANATION OF THE MULTI-HEAD ATTENTION CLASS


**For comparison, the smallest GPT-2 model (117 million parameters) has 12 attention heads and a context vector embedding size of 768.** 

**The largest GPT-2 model (1.5 billion parameters) has 25 attention heads and a context vector embedding size of 1600.**

**Note that the embedding sizes of the token inputs and context embeddings are the same in GPT
models (d_in = d_out).**